In [2]:
# 1. ডেটাসেট লোড
from datasets import load_dataset

raw_datasets = load_dataset("glue", "sst2")
print(raw_datasets)

# 2. tokenizer লোড
from transformers import AutoTokenizer

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# 3. tokenize function (এখানে শুধু একটা sentence)
def tokenize_function_sst2(example):
    # MRPC-তে দুটো ছিল → এখানে শুধু sentence
    return tokenizer(example["sentence"], truncation=True)

# 4. map দিয়ে টোকেনাইজ করা (batched=True দ্রুত করার জন্য)
tokenized_datasets = raw_datasets.map(tokenize_function_sst2, batched=True)

# 5. দেখে নেওয়া যে কী যোগ হয়েছে
print(tokenized_datasets["train"].features)

# 6. Dynamic padding-এর জন্য data collator
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# টেস্ট করে দেখা (যেমন প্রথম ৮টা sample)
samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence"]}
batch = data_collator(samples)

print({k: v.shape for k, v in batch.items()})

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

{'sentence': Value('string'), 'label': ClassLabel(names=['negative', 'positive']), 'idx': Value('int32'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}
{'input_ids': torch.Size([8, 29]), 'token_type_ids': torch.Size([8, 29]), 'attention_mask': torch.Size([8, 29]), 'labels': torch.Size([8])}
